# 02 — Data Cleaning
**Phase 2 — Online Retail Sales Analytics**

Documents all cleaning decisions and validates them against the clean tables.
Actual transforms live in `clean.py`.

In [1]:
import sys
sys.path.insert(0, "..")
import pandas as pd
from src.config import get_engine
pd.set_option("display.max_columns", None)
engine = get_engine()
print("Connected")

Connected


## 1. Before vs After Row Counts

In [2]:
table_pairs = [
    ("orders",        "orders_clean"),
    ("products",      "products_clean"),
    ("customers",     "customers_clean"),
    ("geolocation",   "geolocation_clean"),
    ("order_reviews", "reviews_clean"),
]

import sqlalchemy as sa

print(f"{'Table':<30} {'Raw':>10} {'Clean':>10} {'Removed':>10} {'Retention':>12}")
print("─" * 76)
for raw_t, clean_t in table_pairs:
    with engine.connect() as conn:
        raw_n   = conn.execute(sa.text(f"SELECT COUNT(*) FROM {raw_t}")).scalar()
        clean_n = conn.execute(sa.text(f"SELECT COUNT(*) FROM {clean_t}")).scalar()
    removed = raw_n - clean_n
    pct = clean_n / raw_n * 100
    print(f"  {raw_t:<28} {raw_n:>10,} {clean_n:>10,} {removed:>10,} {pct:>11.4f}%")

Table                                 Raw      Clean    Removed    Retention
────────────────────────────────────────────────────────────────────────────
  orders                           99,441     99,441          0    100.0000%
  products                         32,951     32,951          0    100.0000%
  customers                        99,441     99,441          0    100.0000%
  geolocation                   1,000,163     19,015    981,148      1.9012%
  order_reviews                    99,224     99,224          0    100.0000%


## 2. Cleaning Rule 1 — orders: date columns cast to datetime

In [3]:
orders       = pd.read_sql("SELECT * FROM orders", engine)
orders_clean = pd.read_sql("SELECT * FROM orders_clean", engine)

date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

print("── Raw dtypes ──")
print(orders[date_cols].dtypes.to_string())
print("\n── Clean dtypes ──")
print(orders_clean[date_cols].dtypes.to_string())

── Raw dtypes ──
order_purchase_timestamp         str
order_approved_at                str
order_delivered_carrier_date     str
order_delivered_customer_date    str
order_estimated_delivery_date    str

── Clean dtypes ──
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]


## 3. Cleaning Rule 2 — orders: order_approved_at null backfill

In [4]:
raw_nulls   = orders["order_approved_at"].isnull().sum()
clean_nulls = orders_clean["order_approved_at"].isnull().sum()

print(f"order_approved_at nulls  — raw:   {raw_nulls:,}")
print(f"order_approved_at nulls  — clean: {clean_nulls:,}  ← should be 0")

order_approved_at nulls  — raw:   160
order_approved_at nulls  — clean: 0  ← should be 0


## 4. Cleaning Rule 3 — products: English category join

In [5]:
products       = pd.read_sql("SELECT * FROM products", engine)
products_clean = pd.read_sql("SELECT * FROM products_clean", engine)

print(f"Raw   — has product_category_name_english col: {'product_category_name_english' in products.columns}")
print(f"Clean — has product_category_name_english col: {'product_category_name_english' in products_clean.columns}")
print()
print(f"Clean — 'unknown' category count: {(products_clean['product_category_name_english'] == 'unknown').sum():,}")
print(f"Clean — null category count:      {products_clean['product_category_name_english'].isnull().sum():,}  ← should be 0")

Raw   — has product_category_name_english col: False
Clean — has product_category_name_english col: True

Clean — 'unknown' category count: 623
Clean — null category count:      0  ← should be 0


## 5. Cleaning Rule 4 — customers: duplicate customer_id removal

In [6]:
customers       = pd.read_sql("SELECT * FROM customers", engine)
customers_clean = pd.read_sql("SELECT * FROM customers_clean", engine)

raw_dups   = customers["customer_id"].duplicated().sum()
clean_dups = customers_clean["customer_id"].duplicated().sum()

print(f"Duplicate customer_id — raw:   {raw_dups:,}")
print(f"Duplicate customer_id — clean: {clean_dups:,}  ← should be 0")

Duplicate customer_id — raw:   0
Duplicate customer_id — clean: 0  ← should be 0


## 6. Cleaning Rule 5 — geolocation: one row per ZIP

In [7]:
with engine.connect() as conn:
    import sqlalchemy as sa
    raw_rows   = conn.execute(sa.text("SELECT COUNT(*) FROM geolocation")).scalar()
    clean_rows = conn.execute(sa.text("SELECT COUNT(*) FROM geolocation_clean")).scalar()
    raw_zips   = conn.execute(sa.text("SELECT COUNT(DISTINCT geolocation_zip_code_prefix) FROM geolocation")).scalar()
    clean_zips = conn.execute(sa.text("SELECT COUNT(DISTINCT geolocation_zip_code_prefix) FROM geolocation_clean")).scalar()
    clean_dups = conn.execute(sa.text("""
        SELECT COUNT(*) FROM (
            SELECT geolocation_zip_code_prefix, COUNT(*) AS cnt
            FROM geolocation_clean
            GROUP BY geolocation_zip_code_prefix
            HAVING COUNT(*) > 1
        ) sub
    """)).scalar()

print(f"Raw rows:                 {raw_rows:,}")
print(f"Clean rows:               {clean_rows:,}")
print(f"Distinct ZIPs in raw:     {raw_zips:,}")
print(f"Distinct ZIPs in clean:   {clean_zips:,}")
print(f"Duplicate ZIPs in clean:  {clean_dups:,}  ← should be 0")


Raw rows:                 1,000,163
Clean rows:               19,015
Distinct ZIPs in raw:     19,015
Distinct ZIPs in clean:   19,015
Duplicate ZIPs in clean:  0  ← should be 0


## 7. Cleaning Rule 6 — reviews: null comment fill & date cast

In [8]:
reviews       = pd.read_sql("SELECT * FROM order_reviews", engine)
reviews_clean = pd.read_sql("SELECT * FROM reviews_clean", engine)

print("── Null counts: review_comment_title ──")
print(f"  raw:   {reviews['review_comment_title'].isnull().sum():,}")
print(f"  clean: {reviews_clean['review_comment_title'].isnull().sum():,}  ← should be 0")

print("\n── Null counts: review_comment_message ──")
print(f"  raw:   {reviews['review_comment_message'].isnull().sum():,}")
print(f"  clean: {reviews_clean['review_comment_message'].isnull().sum():,}  ← should be 0")

print("\n── Date column dtypes in clean ──")
print(reviews_clean[["review_creation_date","review_answer_timestamp"]].dtypes.to_string())

── Null counts: review_comment_title ──
  raw:   87,656
  clean: 0  ← should be 0

── Null counts: review_comment_message ──
  raw:   58,247
  clean: 0  ← should be 0

── Date column dtypes in clean ──
review_creation_date       datetime64[us]
review_answer_timestamp    datetime64[us]


## 8. Cleaning Summary

In [9]:
summary = pd.DataFrame([
    {"Table":         "orders",
     "Rule":          "Cast 5 timestamp columns to datetime",
     "Raw Issue":     "string dtype",
     "After Cleaning":"datetime64",
     "Decision":      "Cast"},
    {"Table":         "orders",
     "Rule":          "Backfill order_approved_at nulls",
     "Raw Issue":     f"{orders['order_approved_at'].isnull().sum():,} nulls",
     "After Cleaning":"0 nulls",
     "Decision":      "Fill with order_purchase_timestamp"},
    {"Table":         "products",
     "Rule":          "Join English category name",
     "Raw Issue":     "No English column",
     "After Cleaning":"product_category_name_english added",
     "Decision":      "Left join with category_translation"},
    {"Table":         "products",
     "Rule":          "Fill missing category name",
     "Raw Issue":     f"{products['product_category_name'].isnull().sum():,} nulls",
     "After Cleaning":"'unknown' string",
     "Decision":      "fillna('unknown')"},
    {"Table":         "customers",
     "Rule":          "Remove duplicate customer_id rows",
     "Raw Issue":     f"{customers['customer_id'].duplicated().sum():,} duplicates",
     "After Cleaning":"0 duplicates",
     "Decision":      "drop_duplicates"},
    {"Table":         "geolocation",
     "Rule":          "Deduplicate to one row per ZIP",
     "Raw Issue":     f"{raw_rows - raw_zips:,} duplicate ZIP entries",
     "After Cleaning":"1 row per ZIP",
     "Decision":      "drop_duplicates on zip prefix"},
    {"Table":         "order_reviews",
     "Rule":          "Fill null comment title & message",
     "Raw Issue":     "nulls in comment fields",
     "After Cleaning":"empty string ''",
     "Decision":      "fillna('')"},
    {"Table":         "order_reviews",
     "Rule":          "Cast date columns to datetime",
     "Raw Issue":     "string dtype",
     "After Cleaning":"datetime64",
     "Decision":      "Cast"},
])

summary


,Table,Rule,Raw Issue,After Cleaning,Decision
0,orders,Cast 5 timestamp columns to datetime,string dtype,datetime64,Cast
1,orders,Backfill order_approved_at nulls,160 nulls,0 nulls,Fill with order_purchase_timestamp
2,products,Join English category name,No English column,product_category_name_english added,Left join with category_translation
3,products,Fill missing category name,610 nulls,'unknown' string,fillna('unknown')
4,customers,Remove duplicate customer_id rows,0 duplicates,0 duplicates,drop_duplicates
5,geolocation,Deduplicate to one row per ZIP,"981,148 duplicate ZIP entries",1 row per ZIP,drop_duplicates on zip prefix
6,order_reviews,Fill null comment title & message,nulls in comment fields,empty string '',fillna('')
7,order_reviews,Cast date columns to datetime,string dtype,datetime64,Cast


In [10]:
engine.dispose()
print("Done.")

Done.
